# Grokking regression: single-run training template

This notebook trains one grokking teacher-student regression run and saves:
- benchmark tensors (`X_train.pt`, `y_train.pt`, `X_test.pt`, `y_test.pt`, `teacher_weights.pt`)
- `meta.json` with the run configuration
- `theta_0.pt` (initial student weights)
- `training_history.pt` (train/test loss, accuracy, squared L2 norm)
- model checkpoints every `CHECKPOINT_EVERY` steps

Default full-batch GD settings from the parameter search:
- `ETA = 0.1`
- `LAMBDA_REG = 1e-4` (set to `0.0` to disable weight decay)

To produce the three canonical cases for the paper figures, run this notebook three times:
1. `STUDENT_ALPHA = 2.0`, `LAMBDA_REG = 1e-4`
2. `STUDENT_ALPHA = 2.0`, `LAMBDA_REG = 0.0`
3. `STUDENT_ALPHA = 3.0`, `LAMBDA_REG = 1e-4`

In [ ]:
import os
import json
import math
from pathlib import Path
from typing import List, Dict, Any

import numpy as onp
import torch
import torch.nn as nn
from tqdm.auto import trange

if torch.cuda.is_available():
    TORCH_DEVICE = torch.device("cuda")
else:
    TORCH_DEVICE = torch.device("cpu")
print("Active PyTorch device:", TORCH_DEVICE)

In [ ]:
# ----------------------------
# Hyperparameters and paths
# ----------------------------
ROOT = Path.cwd()

SEED = 421

# Architecture / data
D_IN = 5
D_OUT = 5
WIDTH = 100
TRAIN_SIZE = 100
TEST_SIZE = 100
TEACHER_SEED = 0
STUDENT_SEED = 1
TEACHER_ALPHA = 1.0   # initial teacher weight magnitude
STUDENT_ALPHA = 2.0   # initial student weight magnitude (2.0 or 3.0)

# Optimization
STEPS = 100_000
ETA = 0.1
LAMBDA_REG = 0. #1e-4     # weight regularization (1e-4 with WD, 0.0 without WD)
CHECKPOINT_EVERY = 100
LOG_EVERY = 200
THRESHOLD = 1e-3

CONFIG = {
    "d_in": D_IN,
    "d_out": D_OUT,
    "width": WIDTH,
    "train_size": TRAIN_SIZE,
    "test_size": TEST_SIZE,
    "teacher_seed": TEACHER_SEED,
    "student_seed": STUDENT_SEED,
    "teacher_alpha": TEACHER_ALPHA,
    "student_alpha": STUDENT_ALPHA,
    "num_steps": STEPS,
    "eta": ETA,
    "lambda_reg": LAMBDA_REG,
    "checkpoint_every": CHECKPOINT_EVERY,
    "log_every": LOG_EVERY,
    "threshold": THRESHOLD,
}

RUN_ID = (
    f"din{D_IN}_dout{D_OUT}_w{WIDTH}_nt{TRAIN_SIZE}_alpha{STUDENT_ALPHA}_"
    f"steps{STEPS}_eta{ETA}_reg{LAMBDA_REG}_seed{STUDENT_SEED}"
)

BENCHMARK_DIR = ROOT / "benchmarks" / "omnigrok_regression" / RUN_ID
RESULT_DIR = ROOT / "results" / "omnigrok_regression" / RUN_ID
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Numerical dtype
# ----------------------------
NUMERICAL_DTYPE = "float64"
TORCH_DTYPE = torch.float64 if NUMERICAL_DTYPE == "float64" else torch.float32

onp.random.seed(SEED)
torch.manual_seed(SEED)

print("Numerical dtype:", NUMERICAL_DTYPE)
print("Run ID:", RUN_ID)
print("Benchmark dir:", BENCHMARK_DIR)
print("Result dir:", RESULT_DIR)

In [ ]:
class MLP(nn.Module):
    """5 -> 100 -> 100 -> 5 MLP with tanh activations"""
    def __init__(self, d_in=D_IN, width=WIDTH, d_out=D_OUT):
        super().__init__()
        self.l1 = nn.Linear(d_in, width)
        self.l2 = nn.Linear(width, width)
        self.l3 = nn.Linear(width, d_out)
        self.act = nn.Tanh()

    def forward(self, x):
        x = self.act(self.l1(x))
        x = self.act(self.l2(x))
        return self.l3(x)


@torch.no_grad()
def scale_initialization_(model, alpha):
    """Multiply every initialized parameter by alpha"""
    for p in model.parameters():
        p.mul_(alpha)
    return model


@torch.no_grad()
def parameter_l2_sq(model):
    """Conventional squared L2 norm of all parameters"""
    return sum((p.detach() ** 2).sum() for p in model.parameters()).item()


def per_example_mse(pred, target):
    return ((pred - target) ** 2).mean(dim=1)


def threshold_accuracy(loss_vec, threshold=THRESHOLD):
    return (loss_vec < threshold).float().mean().item()


def count_torch_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
# Generate data on the active PyTorch device
torch.manual_seed(TEACHER_SEED)
onp.random.seed(TEACHER_SEED)

teacher = MLP().to(TORCH_DEVICE).to(TORCH_DTYPE)
scale_initialization_(teacher, TEACHER_ALPHA)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

with torch.no_grad():
    x_train = torch.normal(0.0, 1.0, size=(TRAIN_SIZE, D_IN), device=TORCH_DEVICE, dtype=TORCH_DTYPE)
    y_train = teacher(x_train).detach()
    x_test = torch.normal(0.0, 1.0, size=(TEST_SIZE, D_IN), device=TORCH_DEVICE, dtype=TORCH_DTYPE)
    y_test = teacher(x_test).detach()

# Save benchmark tensors and metadata
torch.save(x_train.cpu(), BENCHMARK_DIR / "X_train.pt")
torch.save(y_train.cpu(), BENCHMARK_DIR / "y_train.pt")
torch.save(x_test.cpu(), BENCHMARK_DIR / "X_test.pt")
torch.save(y_test.cpu(), BENCHMARK_DIR / "y_test.pt")
torch.save(teacher.state_dict(), BENCHMARK_DIR / "teacher_weights.pt")
with open(BENCHMARK_DIR / "meta.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# Save initial student weights
torch.manual_seed(STUDENT_SEED)
student = MLP().to(TORCH_DEVICE).to(TORCH_DTYPE)
scale_initialization_(student, STUDENT_ALPHA)
theta_0_state = student.state_dict()
torch.save(theta_0_state, RESULT_DIR / "theta_0.pt")

print(f"Teacher params: {count_torch_params(teacher):,}")
print(f"Student params: {count_torch_params(student):,}")
print("Train:", x_train.shape, y_train.shape)
print("Test: ", x_test.shape, y_test.shape)

In [ ]:
def train_gd(model, x_train, y_train, x_test, y_test, eta, lambda_reg, n_steps, checkpoint_every, log_every):
    """Full-batch gradient descent on MSE with manual L2 regularization on all parameters"""
    model.train()
    params = list(model.parameters())

    history = {
        "step": [],
        "train_loss": [],
        "test_loss": [],
        "train_acc": [],
        "test_acc": [],
        "l2_sq": [],
    }

    for step in trange(n_steps, desc="GD training"):
        l2_norm_sq = sum(p.pow(2).sum() for p in params)
        reg = 0.5 * lambda_reg * l2_norm_sq

        pred_train = model(x_train)
        train_loss_vec = per_example_mse(pred_train, y_train)
        loss = train_loss_vec.mean() + reg

        if step % log_every == 0:
            with torch.no_grad():
                pred_test = model(x_test)
                test_loss_vec = per_example_mse(pred_test, y_test)

                history["step"].append(step)
                history["train_loss"].append((loss.item() - reg.item()))
                history["test_loss"].append(test_loss_vec.mean().item())
                history["train_acc"].append(threshold_accuracy(train_loss_vec))
                history["test_acc"].append(threshold_accuracy(test_loss_vec))
                history["l2_sq"].append(parameter_l2_sq(model))

        if step % checkpoint_every == 0:
            torch.save(model.state_dict(), RESULT_DIR / f"checkpoint_step_{step}.pt")

        grads = torch.autograd.grad(loss, params, create_graph=False)
        with torch.no_grad():
            for p, g in zip(params, grads):
                p -= eta * g

    # Save final checkpoint
    torch.save(model.state_dict(), RESULT_DIR / f"checkpoint_step_{n_steps}.pt")
    return history


# Re-initialize from theta_0 and train
student_gd = MLP().to(TORCH_DEVICE).to(TORCH_DTYPE)
student_gd.load_state_dict(theta_0_state)

history = train_gd(
    student_gd, x_train, y_train, x_test, y_test,
    eta=ETA, lambda_reg=LAMBDA_REG, n_steps=STEPS,
    checkpoint_every=CHECKPOINT_EVERY, log_every=LOG_EVERY,
)

torch.save(history, RESULT_DIR / "training_history.pt")
print("Saved training history to", RESULT_DIR / "training_history.pt")